# e-stat API取得

`.env` に `E_STAT_APPID`（または `APPID`）を置いておき、以下のセルで読み込み・API呼び出しを行います。

In [ ]:
# 必要ライブラリの読み込み
from dotenv import load_dotenv
import os
import requests
import json

# .env を読み込む（事前に .env に E_STAT_APPID=xxxxx を設定しておく）
load_dotenv()
app_id = os.getenv('E_STAT_APPID') or os.getenv('APPID') or ''
if not app_id:
    raise EnvironmentError('`.env` に E_STAT_APPID または APPID を設定してください')

# ベース URL とパラメータを分けて渡す（requests は str を受け取る）
base_url = 'http://api.e-stat.go.jp/rest/3.0/app/json/getStatsData'
params = {
    'appId': app_id,
    'lang': 'J',
    'statsDataId': '0003045904',
    'metaGetFlg': 'Y',
    'cntGetFlg': 'N',
    'explanationGetFlg': 'Y',
    'annotationGetFlg': 'Y',
    'sectionHeaderFlg': '1',
    'replaceSpChars': '0',
    'cdCatSelect1': 'C01',
    'cdArea': '01000',
}

# リクエスト実行
try:
    resp = requests.get(base_url, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()
except requests.RequestException as e:
    raise SystemExit(f'API request failed: {e}')

# サマリ表示（必要に応じて詳細は data を確認）
print('取得したトップレベルキー:', list(data.keys()))
# 主要部分を見やすく表示
print(json.dumps(data, ensure_ascii=False)[:2000])

取得したトップレベルキー: ['GET_STATS_DATA']
{"GET_STATS_DATA": {"RESULT": {"STATUS": 0, "ERROR_MSG": "正常に終了しました。", "DATE": "2026-06-24T23:19:46.888+09:00"}, "PARAMETER": {"LANG": "J", "STATS_DATA_ID": "0003045904", "NARROWING_COND": {"CODE_AREA_SELECT": "01000"}, "DATA_FORMAT": "J", "START_POSITION": 1, "METAGET_FLG": "Y", "EXPLANATION_GET_FLG": "Y", "ANNOTATION_GET_FLG": "Y", "REPLACE_SP_CHARS": 0, "CNT_GET_FLG": "N", "SECTION_HEADER_FLG": 1}, "STATISTICAL_DATA": {"RESULT_INF": {"TOTAL_NUMBER": 576, "FROM_NUMBER": 1, "TO_NUMBER": 576}, "TABLE_INF": {"@id": "0003045904", "STAT_NAME": {"@code": "00351000", "$": "民間給与実態統計調査"}, "GOV_ORG": {"@code": "00351", "$": "国税庁"}, "STATISTICS_NAME": "民間給与実態統計 結果表", "TITLE": {"@no": "00101", "$": "全国計表　第1表　給与所得者数・給与額・税額 事業所規模別 （2007年～2014年）"}, "CYCLE": "年次", "SURVEY_DATE": 0, "OPEN_DATE": "2015-11-19", "SMALL_AREA": 0, "COLLECT_AREA": "該当なし", "MAIN_CATEGORY": {"@code": "03", "$": "労働・賃金"}, "SUB_CATEGORY": {"@code": "02", "$": "賃金・労働条件"}, "OVERALL_TOTAL_NUMBER":

In [29]:
import pandas as pd
df = pd.DataFrame(data['GET_STATS_DATA']['STATISTICAL_DATA']['DATA_INF']['VALUE'])
display(df)

,@tab,@cat01,@time,@unit,$
0,0010,1,2014000000,人,9825309
1,0010,1,2013000000,人,10025169
2,0010,1,2012000000,人,9713060
3,0010,1,2011000000,人,10010414
4,0010,1,2010000000,人,10303333
...,...,...,...,...,...
571,0090,9,2011000000,千円,148
572,0090,9,2010000000,千円,137
573,0090,9,2009000000,千円,138
574,0090,9,2008000000,千円,157


In [11]:
import pandas as pd
df = pd.DataFrame(data['GET_STATS_DATA']['STATISTICAL_DATA']['DATA_INF']['VALUE'])
display(df)

,@tab,@cat01,@area,@time,@unit,$
0,20200,102,00000,2020100000,人,672323
1,20200,102,01000,2020100000,人,32503
2,20200,102,02000,2020100000,人,8978
3,20200,102,03000,2020100000,人,8754
4,20200,102,04000,2020100000,人,12307
...,...,...,...,...,...,...
3063,20220,273,33100,2020100000,人口10万対,1102.3
3064,20220,273,34100,2020100000,人口10万対,998.1
3065,20220,273,40100,2020100000,人口10万対,1207.2
3066,20220,273,40130,2020100000,人口10万対,1167.3


In [37]:
"""
Utility for calling e-stat '統計表情報取得' (getStatsList) API (仕様 v3.0).

Parameters (日本語説明):
  - app_id: アプリケーションID（必須）。`.env` に `E_STAT_APPID` または `APPID` を設定してください。
  - lang: 取得するデータの言語。'J'（日本語, 省略値）または 'E'（英語）。
  - surveyYears: 調査年月。'yyyy'、'yyyymm'、または 'yyyymm-yyyymm' の形式。
  - openYears: 公開年月。surveyYears と同様の形式。
  - statsField: 統計分野コード（数値2桁/4桁）。
  - statsCode: 政府統計コード（作成機関5桁/政府統計コード8桁）。
  - statsNameList: 統計調査名一覧指定。'Y' を指定すると統計調査名一覧を返す。
  - searchWord: 検索キーワード。AND/OR/NOT を使った複合検索も可（例: '東京 AND 人口'）。
  - searchKind: 検索データ種別。'1'=統計情報（省略値）, '2'=小地域・地域メッシュ。
  - collectArea: 集計地域区分。'1'=全国, '2'=都道府県, '3'=市区町村。
  - explanationGetFlg: 解説情報有無。'Y'=取得（省略値）, 'N'=取得しない。
  - startPosition: データ取得開始位置（1始まり）。継続取得時に使用。
  - limit: 取得件数。省略時はサービス側のデフォルト（例: 統計表検索は100000等）。
  - updatedDate: 更新日付。'yyyy'、'yyyymm'、'yyyymmdd'、または範囲指定。
  - dataFormat: 出力形式。'J'（JSON, 省略値）/'X'（XML）/'C'（CSV）等。
  - callback: JSONP 用のコールバック関数名（JSONP を使う場合のみ）。

注: パラメータは None または空文字の場合リクエストに含めません。
"""
from dotenv import load_dotenv
import os
import requests
import json
from typing import Optional

API_URL = 'https://api.e-stat.go.jp/rest/3.0/app/json/getStatsList'


def _filter_params(d: dict) -> dict:
    return {k: v for k, v in d.items() if v is not None and v != ''}


def get_stats_list(
    app_id: Optional[str],
    lang: Optional[str] = 'J',
    surveyYears: Optional[str] = None,
    openYears: Optional[str] = None,
    statsField: Optional[str] = None,
    statsCode: Optional[str] = None,
    statsNameList: Optional[str] = None,
    searchWord: Optional[str] = None,
    searchKind: Optional[str] = None,
    collectArea: Optional[str] = None,
    explanationGetFlg: Optional[str] = None,
    startPosition: Optional[int] = None,
    limit: Optional[int] = None,
    updatedDate: Optional[str] = None,
    dataFormat: Optional[str] = None,
    callback: Optional[str] = None,
):
    """Call e-stat getStatsList (統計表情報取得).

    All parameters follow the e-stat v3.0 API specification. Any parameter passed as
    None or empty string will be omitted from the request.

    Returns parsed JSON (dict) on success. Raises SystemExit on HTTP/request errors.
    """
    if not app_id:
        raise ValueError('app_id is required (set E_STAT_APPID or APPID in .env)')

    params = {
        'appId': app_id,
        'lang': lang,
        'surveyYears': surveyYears,
        'openYears': openYears,
        'statsField': statsField,
        'statsCode': statsCode,
        'statsNameList': statsNameList,
        'searchWord': searchWord,
        'searchKind': searchKind,
        'collectArea': collectArea,
        'explanationGetFlg': explanationGetFlg,
        'startPosition': startPosition,
        'limit': limit,
        'updatedDate': updatedDate,
        'dataFormat': dataFormat,
        'callback': callback,
    }

    safe_params = _filter_params(params)

    try:
        resp = requests.get(API_URL, params=safe_params, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as e:
        raise SystemExit(f'API request failed: {e}')

    # API JSON endpoint returns JSON structure under GET_STATS_LIST
    try:
        return resp.json()
    except ValueError:
        # fallback: return raw text if JSON parsing fails
        return {'raw': resp.text}


def pretty_print(result: dict, max_chars: int = 4000) -> None:
    s = json.dumps(result, ensure_ascii=False, indent=2)
    print(s[:max_chars])


if __name__ == '__main__':
    # Example usage when run as a script
    load_dotenv()
    app_id = os.getenv('E_STAT_APPID') or os.getenv('APPID')
    if not app_id:
        raise SystemExit('Please set E_STAT_APPID (or APPID) in .env')

    # Minimal example: search for 統計表 containing '人口'
    res = get_stats_list(
        app_id=app_id,
        # searchWord='課税標準額',
        limit=20,
        dataFormat='J',
        statsCode='002005027',
    )
    pretty_print(res)


{
  "GET_STATS_LIST": {
    "RESULT": {
      "STATUS": 102,
      "ERROR_MSG": "政府統計コード（statsCode）の値が正しくありません。",
      "DATE": "2026-06-24T23:37:19.643+09:00"
    },
    "PARAMETER": {
      "LANG": "J",
      "STATS_CODE": "002005027",
      "DATA_FORMAT": "J",
      "LIMIT": 20
    }
  }
}


In [42]:
# 必要ライブラリの読み込み
import pandas as pd

# Pandas 2.2以降で消された applymap を、現行の map に身代わりさせる
if not hasattr(pd.DataFrame, "applymap"):
    pd.DataFrame.applymap = pd.DataFrame.map

import jpstat
from dotenv import load_dotenv
import os
import requests
import json

# .env を読み込む（事前に .env に E_STAT_APPID=xxxxx を設定しておく）
load_dotenv()
app_id = os.getenv('E_STAT_APPID') or os.getenv('APPID') or ''
if not app_id:
    raise EnvironmentError('`.env` に E_STAT_APPID または APPID を設定してください')

stat = jpstat.estat.get_stat(key=app_id)

In [44]:
display(stat)

,@id,STAT_NAME,GOV_ORG
0,00020111,民間企業の勤務条件制度等調査,人事院
1,00020112,国家公務員死因調査,人事院
2,00020131,国家公務員災害補償統計,人事院
3,00020151,退職公務員生活状況調査,人事院
4,00020211,一般職の国家公務員の任用状況調査,人事院
...,...,...,...
289,00650401,家庭からの二酸化炭素排出量の推計に係る実態調査 試験調査,環境省
290,00650402,大気汚染に係る環境保健サーベイランス調査,環境省
291,00650405,食品廃棄物等の発生抑制及び再生利用の促進の取組に係る実態調査,環境省
292,00650408,家庭部門のCO2排出実態統計調査,環境省


In [ ]:
jpstat.options["estat.api_key"] = app_id
data = jpstat.estat.get_data(statsDataId="0000040001", return_note=False)

In [48]:
data

,@unit,産業大分類040001,経営組織040002,全国計040001,時間軸(年次),Value
0,NaN,全産業,総数,全国,1981年,6488329
1,事業所,全産業,民営,全国,1981年,6290703
2,NaN,全産業,個人,全国,1981年,4182274
3,NaN,全産業,法人,全国,1981年,2074479
4,NaN,全産業,会社,全国,1981年,1843464
...,...,...,...,...,...,...
166,NaN,サービス業,公共企業体,全国,1981年,447
167,NaN,サービス業,地方公共団体,全国,1981年,103593
168,NaN,公務,総数,全国,1981年,45765
169,NaN,公務,国,全国,1981年,7841


In [49]:
data = jpstat.estat.get_list(searchWord="人口")

In [50]:
data

,@id,STAT_NAME,GOV_ORG,STATISTICS_NAME,TITLE,SURVEY_DATE,OPEN_DATE,OVERALL_TOTAL_NUMBER
0,0000150041,人口推計,総務省,人口推計 平成5年10月1日現在推計人口,"人口及び人口増加(29),男女別(3)人口数-総人口,日本人人口,外国人人口,全国",199310,2007-10-03,87
1,0000150062,人口推計,総務省,人口推計 平成6年10月1日現在推計人口,"人口及び人口増加(29),男女別(3)人口数-総人口,日本人人口,外国人人口,全国",199410,2007-10-03,87
2,0000150271,人口推計,総務省,人口推計 各年10月1日現在人口 平成17年国勢調査基準 参考表,[参考表]年齢各歳(94)、男女別(3)、人口数、死亡者数、入国超過-総人口、日本人人口(9...,200610,2007-10-03,2538
3,0000150182,人口推計,総務省,人口推計 各年10月1日現在人口 平成12年国勢調査基準 参考表,[参考表]年齢各歳(94)、男女別(3)、人口数、死亡者数、入国超過数-総人口、日本人人口、...,200110,2007-10-03,2538
4,0000150204,人口推計,総務省,人口推計 各年10月1日現在人口 平成12年国勢調査基準 参考表,[参考表]年齢各歳(94)、男女別(3)、人口数、死亡者数、入国超過数-総人口、日本人人口(...,200210,2007-10-03,2538
...,...,...,...,...,...,...,...,...
25011,0003157566,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,351
25012,0003157567,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,273
25013,0003157580,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,273
25014,0003157581,空き家所有者実態調査,国土交通省,平成26年空家実態調査,7 今後5年程度のうちの利用意向・内容及びその理由等 賃貸・売却する上での課題(11区分) ...,201401-201412,2016-12-28,429


In [51]:
data = jpstat.estatFile.get_stat()
data

...............

,@id,STAT_NAME
0,00000001,民間企業退職金実態調査
1,00000002,一般職国家公務員在職状況統計表（人事統計報告）
2,00000003,国家公務員退職手当実態調査
3,00000004,人々のつながりに関する基礎調査
4,00020111,民間企業の勤務条件制度等調査
...,...,...
741,00700010,自衛隊施設（土地）の状況
742,00700011,在日米軍施設・区域の状況
743,00700012,急患輸送実績
744,00700013,緊急発進実施状況


In [54]:
data = jpstat.estatFile.get_list(statsCode="00400001", year="2025")
data

......................

,STAT_NAME,STAT_CAT,SURVEY_DATE,OPEN_DATE,EXCEL,@id,STATISTICS_NAME,PDF
0,学校基本調査,学校基本調査 / （参考資料）年次統計,2025年,2025-12-26,True,000031852301,総括表 1 学校数（昭和２３年～）,NaN
1,学校基本調査,学校基本調査 / （参考資料）年次統計,2025年,2025-12-26,True,000031852302,総括表 2 在学者数（昭和２３年～）,NaN
2,学校基本調査,学校基本調査 / （参考資料）年次統計,2025年,2025-12-26,True,000031852303,総括表 3 教員数（昭和２３年～）,NaN
3,学校基本調査,学校基本調査 / （参考資料）年次統計,2025年,2025-12-26,True,000031852304,総括表 4 進学率（昭和２３年～）,NaN
4,学校基本調査,学校基本調査 / （参考資料）年次統計,2025年,2025-12-26,True,000031852305,総括表 5 卒業者に占める就職者の割合（昭和２５年～）,NaN
...,...,...,...,...,...,...,...,...
1050,学校基本調査,学校基本調査 / 令和７年度 / 高等教育機関 / 学校施設調査,2025年,2025-12-26,True,000040393054,旧報告書掲載集計 115 学校建物の被害等減少の面積,NaN
1051,学校基本調査,学校基本調査 / 令和７年度 / 高等教育機関 / 学校経費調査（令和６年会計年度）,2025年,2025-12-26,True,000040393055,旧報告書掲載集計 116 使途別-学校経費（国・公立大学）,NaN
1052,学校基本調査,学校基本調査 / 令和７年度 / 高等教育機関 / 学校経費調査（令和６年会計年度）,2025年,2025-12-26,True,000040393056,旧報告書掲載集計 117 使途別-学校経費（国・公立大学法人立の高等専門学校等）,NaN
1053,学校基本調査,学校基本調査 / 令和７年度 / 高等教育機関 / 学校経費調査（令和６年会計年度）,2025年,2025-12-26,True,000040393057,旧報告書掲載集計 118 授業料等及び補助金収入,NaN


In [9]:
import requests
from dotenv import load_dotenv
import os

# .env を読み込む（事前に .env に E_STAT_APPID=xxxxx を設定しておく）
load_dotenv()
app_id = os.getenv('E_STAT_APPID') or os.getenv('APPID') or ''
get_stats_list = "https://api.e-stat.go.jp/rest/3.0/app/json/getStatsList"
get_meta_url = "https://api.e-stat.go.jp/rest/3.0/app/json/getMetaInfo"
stats_list_params = {
    "appId": app_id,
    "lang": "J",
}
# meta_params = {
#     "appId": app_id,
#     "lang": "J",
#     "statsDataId": "0003270360",
#     "explanationGetFlg": "Y",
# }
# meta_info = requests.get(get_meta_url, params=params)
# print(meta_info.json())


{'GET_META_INFO': {'RESULT': {'STATUS': 0, 'ERROR_MSG': '正常に終了しました。', 'DATE': '2026-07-06T23:08:42.174+09:00'}, 'PARAMETER': {'LANG': 'J', 'STATS_DATA_ID': '0003270360', 'EXPLANATION_GET_FLG': 'Y', 'DATA_FORMAT': 'J'}, 'METADATA_INF': {'TABLE_INF': {'@id': '0003270360', 'STAT_NAME': {'@code': '00552010', '$': '知的財産活動調査'}, 'GOV_ORG': {'@code': '00552', '$': '特許庁'}, 'STATISTICS_NAME': '知的財産活動調査', 'TITLE': {'@no': '1-6', '$': '【更新終了】業種別出願件数階級別（平成18年度［2006年度］まで） 業種別出願件数階級別の産業財産権制度の利用状況について 特許出願又は審査請求－国内出願'}, 'CYCLE': '年度次', 'SURVEY_DATE': 0, 'OPEN_DATE': '2022-01-11', 'SMALL_AREA': 0, 'COLLECT_AREA': '該当なし', 'MAIN_CATEGORY': {'@code': '11', '$': '情報通信・科学技術'}, 'SUB_CATEGORY': {'@code': '03', '$': '知的財産'}, 'OVERALL_TOTAL_NUMBER': 3420, 'UPDATED_DATE': '2026-07-03', 'STATISTICS_NAME_SPEC': {'TABULATION_CATEGORY': '知的財産活動調査'}, 'DESCRIPTION': '', 'TITLE_SPEC': {'TABLE_CATEGORY': '【更新終了】業種別出願件数階級別（平成18年度［2006年度］まで）', 'TABLE_NAME': '業種別出願件数階級別の産業財産権制度の利用状況について', 'TABLE_SUB_CATEGORY1': '特許出願又は審査請求

In [11]:
import os
import time
import json
import requests
from dotenv import load_dotenv

# .envファイルを読み込む
load_dotenv()
APP_ID = os.getenv('E_STAT_APPID') or os.getenv('APPID') or ''

if not APP_ID:
    raise ValueError("エラー: .env ファイルに E_STAT_APPID または APPID が設定されていません。")

# e-Stat API エンドポイント
URL_GET_STATS_LIST = "https://api.e-stat.go.jp/rest/3.0/app/json/getStatsList"
URL_GET_META_INFO = "https://api.e-stat.go.jp/rest/3.0/app/json/getMetaInfo"

# 設定パラメータ
SAVE_DIR = "./estat_metadata"
MAX_ITEMS_PER_FILE = 5000  # 1ファイルあたり約5000件（約7.5MB〜10MB）で分割保存
INTERVAL_SEC = 1.0         # APIサーバーに負荷をかけないための待機時間（秒）


def fetch_all_stats_ids():
    """1. すべての統計表ID（statsDataId）を取得する"""
    print("すべての統計表IDを検索中...")
    stats_ids = []
    start_position = 1
    page_size = 500
    max_retries = 3

    while True:
        params = {
            "appId": APP_ID,
            "lang": "J",
            "startPosition": start_position,
            "limit": page_size,
        }

        response = None
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(URL_GET_STATS_LIST, params=params, timeout=60)
                response.raise_for_status()
                res_json = response.json()
                break
            except (requests.exceptions.RequestException, json.JSONDecodeError) as e:
                if attempt < max_retries:
                    wait_time = 2 ** attempt
                    print(f"  [リトライ] start={start_position} 試行 {attempt}/{max_retries} 失敗: {e}。{wait_time}秒後に再試行します。")
                    time.sleep(wait_time)
                else:
                    print(f"  [エラー] start={start_position} 取得失敗: {e}")
                    return stats_ids

        result_status = res_json.get("GET_STATS_LIST", {}).get("RESULT", {})
        if result_status.get("STATUS") != 0:
            raise RuntimeError(f"e-Stat API エラー: {result_status.get('ERROR_MSG')}")

        stats_list = res_json.get("GET_STATS_LIST", {}).get("DATALIST_INF", {}).get("TABLE_INF", [])
        if isinstance(stats_list, dict):
            stats_list = [stats_list]

        if not stats_list:
            break

        for table in stats_list:
            if "@id" in table:
                stats_ids.append(table["@id"])

        print(f"  start={start_position} 取得件数: {len(stats_list)}")
        if len(stats_list) < page_size:
            break

        start_position += len(stats_list)
        time.sleep(INTERVAL_SEC)

    print(f"合計 {len(stats_ids)} 件の統計表IDを取得しました。")
    return stats_ids


def fetch_and_save_metadata(stats_ids):
    """2. 各統計表IDのメタデータを取得し、分割してファイルに保存する"""
    if not stats_ids:
        print("処理する統計表IDがありません。終了します。")
        return

    os.makedirs(SAVE_DIR, exist_ok=True)
    
    current_chunk = []
    file_counter = 1
    total_processed = 0
    
    print(f"メタデータの取得を開始します。保存先: {SAVE_DIR}/")

    for idx, stats_id in enumerate(stats_ids):
        params = {
            "appId": APP_ID,
            "lang": "J",
            "statsDataId": stats_id,
            "explanationGetFlg": "Y"
        }
        
        # リトライアルゴリズム（最大3回試行）
        meta_data = None
        for attempt in range(3):
            try:
                response = requests.get(URL_GET_META_INFO, params=params, timeout=15)
                response.raise_for_status()
                res_json = response.json()
                
                # e-Statの固有エラーチェック
                result_status = res_json.get("GET_META_INFO", {}).get("RESULT", {})
                if result_status.get("STATUS") == 0:
                    meta_data = res_json.get("GET_META_INFO", {}).get("METADATA_INF", {})
                    break
                else:
                    print(f" [警告] ID:{stats_id} のメタデータ取得失敗 (APIメッセージ: {result_status.get('ERROR_MSG')})")
                    break # 固有エラー（データが存在しない等）の場合はリトライしない
                    
            except (requests.exceptions.RequestException, json.JSONDecodeError) as e:
                print(f" [バックオフ] ID:{stats_id} 試行 {attempt + 1}/3 失敗: {e}")
                time.sleep(2 * (attempt + 1)) # 失敗するごとに待機時間を伸ばす
        
        # 正常に取得できた場合、リストに追加
        if meta_data:
            current_chunk.append({
                "statsDataId": stats_id,
                "metadata": meta_data
            })
            total_processed += 1

        # 指定の件数に達するか、最後の要素になったらファイルに書き出し
        if len(current_chunk) >= MAX_ITEMS_PER_FILE or (idx == len(stats_ids) - 1 and current_chunk):
            file_path = os.path.join(SAVE_DIR, f"metadata_chunk_{file_counter:03d}.json")
            try:
                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(current_chunk, f, ensure_ascii=False, indent=2)
                print(f"【保存完了】 {file_path} に {len(current_chunk)} 件のメタデータを保存しました。")
                file_counter += 1
                current_chunk = [] # チャンクを初期化
            except IOError as e:
                print(f"❌ ファイル保存エラー ({file_path}): {e}")

        # サーバー負荷軽減のためのウェイト
        time.sleep(INTERVAL_SEC)
        
        # 進捗を定期的に表示
        if (idx + 1) % 10 == 0 or (idx + 1) == len(stats_ids):
            print(f"進捗: {idx + 1}/{len(stats_ids)} 件処理完了 (成功: {total_processed}件)")

    print(f"すべての処理が完了しました。総成功件数: {total_processed}件")


if __name__ == "__main__":
    # 1. 統計表IDを全件取得
    all_ids = fetch_all_stats_ids()
    
    # 2. メタデータを個別取得して分割保存
    fetch_and_save_metadata(all_ids)

すべての統計表IDを検索中...
  start=1 取得件数: 500
  start=501 取得件数: 500
  start=1001 取得件数: 500
  start=1501 取得件数: 500
  start=2001 取得件数: 500
  start=2501 取得件数: 500
  start=3001 取得件数: 500
  start=3501 取得件数: 500
  start=4001 取得件数: 500
  start=4501 取得件数: 500
  start=5001 取得件数: 500
  start=5501 取得件数: 500
  start=6001 取得件数: 500
  start=6501 取得件数: 500
  start=7001 取得件数: 500
  start=7501 取得件数: 500
  start=8001 取得件数: 500
  start=8501 取得件数: 500
  start=9001 取得件数: 500
  start=9501 取得件数: 500
  start=10001 取得件数: 500
  start=10501 取得件数: 500
  start=11001 取得件数: 500
  start=11501 取得件数: 500
  start=12001 取得件数: 500
  start=12501 取得件数: 500
  start=13001 取得件数: 500
  start=13501 取得件数: 500
  start=14001 取得件数: 500
  start=14501 取得件数: 500
  start=15001 取得件数: 500
  start=15501 取得件数: 500
  start=16001 取得件数: 500
  start=16501 取得件数: 500
  start=17001 取得件数: 500
  start=17501 取得件数: 500
  start=18001 取得件数: 500
  start=18501 取得件数: 500
  start=19001 取得件数: 500
  start=19501 取得件数: 500
  start=20001 取得件数: 500
  start=20501 取得件数: 500